# Fine-tune Qwen3-4B: SFT (DoRA+NEFTune) -> ORPO — truyện ngụ ngôn tiếng Việt
Chạy trên Colab T4. Thứ tự: cài đặt -> hyperparams -> nạp data -> nạp model -> SFT -> sinh preference -> ORPO -> sinh thử -> export GGUF.

In [ ]:
%pip install -q unsloth
%pip uninstall -q -y torchao
print("cài đặt xong (đã gỡ torchao tránh xung đột)")

In [ ]:
MODEL_NAME      = "unsloth/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH  = 2048
LOAD_IN_4BIT    = True
# LoRA + DoRA
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.0
USE_DORA        = True
# SFT
SFT_LR          = 5e-5
SFT_EPOCHS      = 2
NEFTUNE_ALPHA   = 5
BATCH_SIZE      = 1
GRAD_ACCUM      = 8
WARMUP_STEPS    = 5
SEED            = 42
# ORPO
ORPO_LR         = 8e-6
ORPO_BETA       = 0.1
ORPO_EPOCHS     = 1
ORPO_MAX_LEN    = 2048      # hạ 1024 nếu OOM
ORPO_PROMPT_LEN = 1024
# Data / format
TRAIN_PATH      = "train.jsonl"
VAL_PATH        = "val.jsonl"
SYSTEM_PROMPT   = "Bạn là người kể truyện ngụ ngôn cho trẻ em."
ENABLE_THINKING = False
MERGED_DIR      = "qwen3-4b-fable-merged"
print("hyperparams:", MODEL_NAME, "| SFT", SFT_LR, SFT_EPOCHS, "| DoRA", USE_DORA, "| ORPO", ORPO_LR)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=LOAD_IN_4BIT)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED, use_dora=USE_DORA)
print("model loaded, DoRA =", USE_DORA)

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": TRAIN_PATH, "val": VAL_PATH})
def to_text(ex):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":ex["instruction"]},
            {"role":"assistant","content":ex["output"]}]
    try: t = tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=ENABLE_THINKING)
    except TypeError: t = tokenizer.apply_chat_template(msgs, tokenize=False)
    return {"text": t}
sft_ds = ds.map(to_text)
print(sft_ds)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=sft_ds["train"], eval_dataset=sft_ds["val"],
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS, num_train_epochs=SFT_EPOCHS, learning_rate=SFT_LR,
        logging_steps=5, eval_strategy="epoch", output_dir="sft_out",
        dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, seed=SEED,
        neftune_noise_alpha=NEFTUNE_ALPHA),
)
trainer = train_on_responses_only(
    trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

In [ ]:
import json
# rejected = sinh bằng model NỀN (tắt adapter) để có "output base"
FastLanguageModel.for_inference(model)
rows = [json.loads(l) for l in open(TRAIN_PATH, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r.get("type") == "story"]
pref = []
with model.disable_adapter():           # tắt LoRA -> hành vi base
    for r in rows:
        msgs = [{"role":"system","content":SYSTEM_PROMPT},
                {"role":"user","content":r["instruction"]}]
        try: p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
        except TypeError: p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tokenizer(p, return_tensors="pt").to("cuda")
        out = model.generate(**inp, max_new_tokens=400, do_sample=True, temperature=0.8,
                             top_p=0.9, repetition_penalty=1.3, no_repeat_ngram_size=3)
        rejected = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        pref.append({"prompt": p, "chosen": r["output"], "rejected": rejected})
with open("preference.jsonl","w",encoding="utf-8") as f:
    for x in pref: f.write(json.dumps(x, ensure_ascii=False)+"\n")
print("preference pairs:", len(pref))
print("VD rejected[:200]:", pref[0]["rejected"][:200])

In [ ]:
from datasets import load_dataset
from trl import ORPOTrainer, ORPOConfig
from unsloth import FastLanguageModel
FastLanguageModel.for_training(model)     # bật lại chế độ train
pref_ds = load_dataset("json", data_files={"train": "preference.jsonl"})["train"]
orpo = ORPOTrainer(
    model=model,
    args=ORPOConfig(
        beta=ORPO_BETA, learning_rate=ORPO_LR, num_train_epochs=ORPO_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        max_length=ORPO_MAX_LEN, max_prompt_length=ORPO_PROMPT_LEN,
        logging_steps=5, output_dir="orpo_out", seed=SEED, warmup_steps=WARMUP_STEPS),
    train_dataset=pref_ds, processing_class=tokenizer,
)
orpo.train()

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":"Viết một truyện ngụ ngôn cho trẻ em về chủ đề: lòng kiên nhẫn. Bài học đạo đức: kiên nhẫn sẽ thành công. Độ tuổi phù hợp: 6-8 tuổi."}]
try: p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
except TypeError: p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inp = tokenizer(p, return_tensors="pt").to("cuda")
out = model.generate(**inp, max_new_tokens=400, do_sample=True, temperature=0.8, top_p=0.9,
                     repetition_penalty=1.3, no_repeat_ngram_size=3)
print(tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
model.save_pretrained_gguf(MERGED_DIR, tokenizer, quantization_method="q8_0")
import glob, os
g = sorted(glob.glob("**/*.gguf", recursive=True), key=lambda p: os.path.getsize(p), reverse=True)
print("GGUF:", [(c, round(os.path.getsize(c)/1e6,1),"MB") for c in g])
from google.colab import files; files.download(g[0])